In [1]:
import pandas as pd
import numpy as np
import os
import gc
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

BASE_DIR = ".."
DATA_PATH = os.path.join(BASE_DIR, "data")
SUB_PATH = os.path.join(BASE_DIR, "submissions")

def reduce_mem(df):
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = df[col].astype('float32')
        if df[col].dtype == 'int64':
            df[col] = df[col].astype('int32')
    return df

print("Cell 1: Siap Tempur!")

Cell 1: Siap Tempur!


In [2]:
df_train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
df_test = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

def align_data(df):
    # Ekstrak angka dari battery
    df['Battery_Level'] = df['Battery_Level'].astype(str).str.extract('(\d+)').astype(float)
    # Standarisasi Signal
    df['Signal_Strength'] = df['Signal_Strength'].astype(str).apply(
        lambda x: 'Low' if x in ['Edge', '3G', 'None'] else 'High'
    )
    return reduce_mem(df)

df_train = align_data(df_train)
df_test = align_data(df_test)
print("Cell 2: Data Aligned!")

Cell 2: Data Aligned!


In [4]:
def build_hero_features(df):
    # 1. Waktu
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df['hour'] = df['Timestamp'].dt.hour
    
    # 2. Tech Interaction
    df['nav_stress'] = ((df['GPS_Accuracy_M'] > 20) & (df['Signal_Strength'] == 'Low')).astype(int)
    
    # 3. Efisiensi Rute
    lat_diff = (df['Dropoff_Lat'] - df['Pickup_Lat']).abs()
    long_diff = (df['Dropoff_Long'] - df['Pickup_Long']).abs()
    euclidean = np.sqrt(lat_diff**2 + long_diff**2)
    df['route_anomaly'] = df['Distance_KM'] / (euclidean + 0.005)
    
    # 4. Sensor Magnitude
    df['accel_mag'] = np.sqrt(df['Accel_X']**2 + df['Accel_Y']**2 + df['Accel_Z']**2)
    
    # --- PEMBERSIHAN TOTAL ---
    # List ini hanya berisi kolom yang PASTI ada di dataset train/test
    cols_to_keep = [
        'hour', 'nav_stress', 'route_anomaly', 'accel_mag', 
        'Distance_KM', 'Est_Price_IDR', 'GPS_Accuracy_M',
        'Pickup_Lat', 'Pickup_Long', 'Dropoff_Lat', 'Dropoff_Long',
        'Signal_Strength', 'Battery_Level'
    ]
    
    # Return hanya kolom yang ada di cols_to_keep
    return df[cols_to_keep]

y = df_train['Trip_Label']
X = build_hero_features(df_train)
X_test_final = build_hero_features(df_test)

del df_train
gc.collect()
print("Cell 3: Fix! Avg_Speed dihapus, kolom sudah sinkron.")

Cell 3: Fix! Avg_Speed dihapus, kolom sudah sinkron.


In [5]:
# Ambil 10%
X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=0.10, stratify=y, random_state=42)

del X, y
gc.collect()

# Pastikan Signal_Strength jadi string
cat_features = ['Signal_Strength']
for col in cat_features:
    X_train[col] = X_train[col].fillna("UNK").astype(str)
    X_val[col] = X_val[col].fillna("UNK").astype(str)
    X_test_final[col] = X_test_final[col].fillna("UNK").astype(str)

# Double check: Jika masih ada kolom object lain, paksa jadi float atau drop
# Ini untuk mencegah error "Cannot convert string to float" lagi
for col in X_train.columns:
    if col not in cat_features and X_train[col].dtype == 'object':
        print(f"Menghapus kolom siluman: {col}")
        X_train.drop(columns=[col], inplace=True)
        X_val.drop(columns=[col], inplace=True)
        X_test_final.drop(columns=[col], inplace=True)

gc.collect()
print("Cell 4: Verifikasi Tipe Data Selesai!")

Cell 4: Verifikasi Tipe Data Selesai!


/tmp/ipykernel_13269/2446238724.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test_final[col] = X_test_final[col].fillna("UNK").astype(str)


In [6]:
model = CatBoostClassifier(
    iterations=500,       # Persingkat lagi
    learning_rate=0.1,
    depth=4,              # Tetap di depth 4 agar laptop tidak freeze
    task_type="CPU",
    thread_count=10,      
    loss_function='MultiClass',
    # Gunakan class_weights manual daripada 'Balanced' agar lebih stabil di RAM
    class_weights={'Perfect_Trip': 1, 'Safety_Violation': 2, 'Fraud_Indication': 5, 
                   'Navigation_Issue': 10, 'Service_Complaint': 10},
    early_stopping_rounds=30,
    random_seed=42,
    verbose=50
)

model.fit(X_train, y_train, cat_features=cat_features, eval_set=(X_val, y_val), use_best_model=True)

0:	learn: 1.5296838	test: 1.5297376	best: 1.5297376 (0)	total: 387ms	remaining: 3m 13s
50:	learn: 1.1080917	test: 1.1088535	best: 1.1088535 (50)	total: 11.8s	remaining: 1m 44s
100:	learn: 1.1017847	test: 1.1029622	best: 1.1029622 (100)	total: 23.3s	remaining: 1m 31s
150:	learn: 1.0993608	test: 1.1009818	best: 1.1009818 (150)	total: 34.7s	remaining: 1m 20s
200:	learn: 1.0976096	test: 1.0997940	best: 1.0997940 (200)	total: 45.8s	remaining: 1m 8s
250:	learn: 1.0963813	test: 1.0991155	best: 1.0991155 (250)	total: 56.7s	remaining: 56.2s
300:	learn: 1.0956665	test: 1.0989812	best: 1.0989722 (293)	total: 1m 7s	remaining: 44.6s
350:	learn: 1.0947616	test: 1.0985737	best: 1.0985725 (348)	total: 1m 18s	remaining: 33.2s
400:	learn: 1.0941428	test: 1.0984520	best: 1.0984511 (399)	total: 1m 29s	remaining: 22s
Stopped by overfitting detector  (30 iterations wait)

bestTest = 1.098443161
bestIteration = 414

Shrink model to first 415 iterations.


In [7]:
probs = model.predict_proba(X_val)
class_names = model.classes_

# PENGALI EKSTREM: Kita paksa model lari ke kelas minoritas
multipliers = {
    'Navigation_Issue': 3.5,  # Naik drastis dari 1.4
    'Service_Complaint': 3.5, # Naik drastis dari 1.4
    'Fraud_Indication': 1.5,
    'Perfect_Trip': 0.8,      # Kita tekan kelas mayoritas
    'Safety_Violation': 1.1
}

for i, label in enumerate(class_names):
    if label in multipliers:
        probs[:, i] *= multipliers[label]

y_val_pred = [class_names[np.argmax(p)] for p in probs]

print("REKOR BARU MACRO F1:")
print(f"Macro F1 Score: {f1_score(y_val, y_val_pred, average='macro'):.4f}")
print("-" * 50)
print(classification_report(y_val, y_val_pred))

REKOR BARU MACRO F1:
Macro F1 Score: 0.3744
--------------------------------------------------


/home/mmarufqk/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mmarufqk/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


                   precision    recall  f1-score   support

 Fraud_Indication       0.92      0.97      0.94    360282
 Navigation_Issue       0.12      0.53      0.19    721611
     Perfect_Trip       0.00      0.00      0.00   3957846
 Safety_Violation       0.94      0.39      0.56   1441436
Service_Complaint       0.11      0.46      0.18    718825

         accuracy                           0.23   7200000
        macro avg       0.42      0.47      0.37   7200000
     weighted avg       0.26      0.23      0.20   7200000



/home/mmarufqk/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [8]:
test_probs = model.predict_proba(X_test_final)
for i, label in enumerate(class_names):
    if label in multipliers:
        test_probs[:, i] *= multipliers[label]

test_preds = [class_names[np.argmax(p)] for p in test_probs]

submission = pd.DataFrame({"Trip_ID": df_test["Trip_ID"], "Trip_Label": test_preds})
submission.to_csv(os.path.join(SUB_PATH, "submission_hero_07.csv"), index=False)
print("Bismillah! Cek Kaggle sekarang, bos ku!")

Bismillah! Cek Kaggle sekarang, bos ku!
